In [0]:
# ==============================================================================
# DOCUMENTAÇÃO DO NOTEBOOK
# Nome: 00_lnd_ingestion_forms
# Camada: LND (Landing Zone)
# Descrição: 
#   Lê a planilha pública do Google Forms via URL, higieniza os cabeçalhos 
#   e salva os dados brutos na Landing Zone de forma INCREMENTAL.
#   Valida se a tabela já existe: se sim, cruza pelo "CARIMBO_DE_DATA_HORA" 
#   e insere (append) estritamente as linhas novas.
# ==============================================================================

In [0]:
import pandas as pd
import re
from pyspark.sql import SparkSession

In [0]:
catalogo = "Projeto_i30"
esquema = "lnd"
tabela_lnd = f"{catalogo}.{esquema}.forms_raw"

url = "https://docs.google.com/spreadsheets/d/1d7JGZG_Mc4DfQFOGBbdw-g5Rn0WTpYtL8nyQm4YVlc8/export?format=csv"


In [0]:
# 1. Cria a estrutura caso não exista
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalogo}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{esquema}")

In [0]:
# 2. Puxa os dados novos do Forms e higieniza colunas
print("Baixando dados brutos do Google Forms...")
pdf_raw = pd.read_csv(url)
pdf_raw.columns = [re.sub(r'[ ,;{}()\n\t=/-]', '_', col).strip('_').upper() for col in pdf_raw.columns]
df_spark_novo = spark.createDataFrame(pdf_raw)

In [0]:
# 3. Lógica Incremental Inteligente
if spark.catalog.tableExists(tabela_lnd):
    print("Tabela LND encontrada! Verificando se há dados novos...")
    
    # Lê a tabela atual do Databricks
    df_atual = spark.table(tabela_lnd)
    
    # Compara a tabela nova com a atual usando o CARIMBO_DE_DATA_HORA (Left Anti Join)
    # Ele separa estritamente os carimbos que vieram do Google e que AINDA NÃO estão no Databricks
    df_diferenca = df_spark_novo.join(df_atual, on="CARIMBO_DE_DATA_HORA", how="left_anti")
    
    qtd_novos = df_diferenca.count()
    
    if qtd_novos > 0:
        print(f"Encontrados {qtd_novos} registros novos! Fazendo o append...")
        (
            df_diferenca.write
            .format("delta")
            .mode("append")
            .saveAsTable(tabela_lnd)
        )
    else:
        print("A tabela já está 100% igual ao Google Forms. Nenhuma linha nova adicionada para evitar repetição.")

else:
    # Se a tabela não existir, faz a primeira carga completa (Carga Inicial)
    print("Primeira execução: Criando a tabela e fazendo a carga inicial...")
    (
        df_spark_novo.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_lnd)
    )

print(f"\nFinalizado! Visão atual da tabela {tabela_lnd}:")

In [0]:
display(spark.table(tabela_lnd))